# Experiment 7 — Soft Actor-Critic (SAC)

A cleaned and personalized SAC implementation for `Pendulum-v1`.

**Focus:** twin critics, stochastic actor, automatic entropy tuning, replay buffer, and soft target updates.

In [ ]:
# Install once in a fresh notebook environment if needed
# %pip install torch gymnasium numpy scipy matplotlib -q

import random
from collections import deque

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
HIDDEN = 128
BATCH_SIZE = 128
GAMMA = 0.99
TAU = 0.005
LR = 3e-4

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN), nn.ReLU()
        )
        self.mean = nn.Linear(HIDDEN, action_dim)
        self.log_std = nn.Linear(HIDDEN, action_dim)
        self.max_action = max_action

    def sample(self, state):
        features = self.backbone(state)
        mean = self.mean(features)
        log_std = self.log_std(features).clamp(-20, 2)
        dist = torch.distributions.Normal(mean, log_std.exp())
        z = dist.rsample()
        squashed = torch.tanh(z)
        action = squashed * self.max_action
        log_prob = (dist.log_prob(z) - torch.log(1 - squashed.pow(2) + 1e-6)).sum(-1, keepdim=True)
        return action, log_prob

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, 1)
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=-1))

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch)
        return (
            torch.tensor(np.asarray(state), dtype=torch.float32),
            torch.tensor(np.asarray(action), dtype=torch.float32),
            torch.tensor(np.asarray(reward), dtype=torch.float32).unsqueeze(1),
            torch.tensor(np.asarray(next_state), dtype=torch.float32),
            torch.tensor(np.asarray(done), dtype=torch.float32).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buffer)

class SACAgent:
    def __init__(self, state_dim, action_dim, max_action):
        self.actor = Actor(state_dim, action_dim, max_action)
        self.q1 = Critic(state_dim, action_dim)
        self.q2 = Critic(state_dim, action_dim)
        self.target_q1 = Critic(state_dim, action_dim)
        self.target_q2 = Critic(state_dim, action_dim)
        self.target_q1.load_state_dict(self.q1.state_dict())
        self.target_q2.load_state_dict(self.q2.state_dict())

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=LR)
        self.q1_optimizer = optim.Adam(self.q1.parameters(), lr=LR)
        self.q2_optimizer = optim.Adam(self.q2.parameters(), lr=LR)
        self.log_alpha = torch.zeros(1, requires_grad=True)
        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=LR)
        self.target_entropy = -float(action_dim)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def act(self, state):
        with torch.no_grad():
            action, _ = self.actor.sample(torch.tensor(state, dtype=torch.float32).unsqueeze(0))
        return action.squeeze(0).numpy()

    def update(self, buffer):
        if len(buffer) < BATCH_SIZE:
            return
        state, action, reward, next_state, done = buffer.sample(BATCH_SIZE)

        with torch.no_grad():
            next_action, next_log_prob = self.actor.sample(next_state)
            next_q = torch.min(self.target_q1(next_state, next_action), self.target_q2(next_state, next_action)) - self.alpha * next_log_prob
            target = reward + (1 - done) * GAMMA * next_q

        q1_loss = nn.functional.mse_loss(self.q1(state, action), target)
        self.q1_optimizer.zero_grad(); q1_loss.backward(); self.q1_optimizer.step()
        q2_loss = nn.functional.mse_loss(self.q2(state, action), target)
        self.q2_optimizer.zero_grad(); q2_loss.backward(); self.q2_optimizer.step()

        new_action, log_prob = self.actor.sample(state)
        q_min = torch.min(self.q1(state, new_action), self.q2(state, new_action))
        actor_loss = (self.alpha.detach() * log_prob - q_min).mean()
        self.actor_optimizer.zero_grad(); actor_loss.backward(); self.actor_optimizer.step()

        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad(); alpha_loss.backward(); self.alpha_optimizer.step()

        for source, target_net in ((self.q1, self.target_q1), (self.q2, self.target_q2)):
            for src_param, tgt_param in zip(source.parameters(), target_net.parameters()):
                tgt_param.data.mul_(1 - TAU).add_(TAU * src_param.data)

In [ ]:
env = gym.make('Pendulum-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])
agent = SACAgent(state_dim, action_dim, max_action)
buffer = ReplayBuffer()
reward_history = []

NUM_EPISODES = 50
for episode in range(NUM_EPISODES):
    state, _ = env.reset(seed=SEED + episode)
    done = False
    total_reward = 0.0
    while not done:
        action = env.action_space.sample() if len(buffer) < 500 else agent.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.add(state, action, reward, next_state, float(done))
        agent.update(buffer)
        state = next_state
        total_reward += reward
    reward_history.append(total_reward)
    print(f'Episode {episode + 1:02d}: reward={total_reward:7.1f}, alpha={agent.alpha.item():.3f}')

last_rewards = reward_history[-10:]
print(f'\nFinal performance (last {len(last_rewards)} episodes): {np.mean(last_rewards):.1f} +/- {stats.sem(last_rewards):.1f} (mean +/- SEM)')
env.close()

## Result
The notebook prints the reward for every episode and a final mean ± SEM over the last 10 episodes.

> Note: RL rewards are stochastic, so your exact numbers will change across runs. The important check is that the notebook executes cleanly and reports the training metrics.